In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU count     :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}  VRAM={props.total_memory/1e9:.1f} GB')


CUDA available: True
GPU count     : 2
  GPU 0: Tesla T4  VRAM=15.6 GB
  GPU 1: Tesla T4  VRAM=15.6 GB


In [2]:
import os, re, gc, random, warnings, xml.etree.ElementTree as ET
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
N_GPUS = torch.cuda.device_count()
print('Torch:', torch.__version__, '| GPUs:', N_GPUS)


Torch: 2.10.0+cu128 | GPUs: 2


In [3]:
COMPETITION_DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline'),
    Path('/kaggle/input/svg-generation'),
    Path('/kaggle/input/competitions/dl-spring-2026-svg-generation'),
]
for p in COMPETITION_DATA_CANDIDATES:
    if (p / 'test.csv').exists():
        COMPETITION_DATA_DIR = p
        break
else:
    raise FileNotFoundError('Could not find test.csv')

MODEL_CANDIDATES = [
    Path('/kaggle/input/qwen-svg-merged-model/merged_model'),
    Path('/kaggle/input/qwen-svg-merged-model/kaggle/working/merged_model'),
    Path('/kaggle/input/qwen-svg-merged-model'),
    Path('/kaggle/input/datasets/sharvingavad0527/mergedmodelfinal'),
]
for p in MODEL_CANDIDATES:
    if p.exists() and (p / 'config.json').exists():
        MODEL_DIR = p
        break
else:
    raise FileNotFoundError('Could not find merged model config.json')

CONFIG = {
    'model_dir':            str(MODEL_DIR),
    'competition_data_dir': str(COMPETITION_DATA_DIR),

    # Prompts are 800-900 chars (~230-255 tokens). 256 was cutting them off!
    # 512 gives comfortable headroom for the system prompt + full user prompt.
    'max_input_length':     512,

    # Must match training (seq_len=2048, SVG truncation at ~5500 chars = ~1700 tokens)
    'max_new_tokens':       1800,

    # Safe for 2xT4 with 1800 new tokens
    'inference_batch_size': 6,

    'submission_path':      '/kaggle/working/submission.csv',
}

print('Model dir       :', CONFIG['model_dir'])
print('max_input_length:', CONFIG['max_input_length'])
print('max_new_tokens  :', CONFIG['max_new_tokens'])
print('batch_size      :', CONFIG['inference_batch_size'])


Model dir       : /kaggle/input/datasets/sharvingavad0527/mergedmodelfinal
max_input_length: 512
max_new_tokens  : 1800
batch_size      : 6


In [4]:
# ── Constants ──────────────────────────────────────────────────────────────
ALLOWED_TAGS = {
    'svg','g','path','rect','circle','ellipse','line','polyline','polygon',
    'defs','use','symbol','clipPath','mask','linearGradient','radialGradient',
    'stop','text','tspan','title','desc','style','pattern','marker','filter',
}
FALLBACK_SVG = (
    "<svg xmlns='http://www.w3.org/2000/svg' width='256' height='256' viewBox='0 0 256 256'>"
    "<rect x='32' y='32' width='192' height='192' rx='16' fill='#d1d5db'/>"
    "<circle cx='128' cy='128' r='48' fill='#6b7280'/></svg>"
)
SVG_REGEX = re.compile(r'<svg[\s\S]*?</svg>', flags=re.IGNORECASE)


def normalize_svg_root(svg: str) -> str:
    """
    Critical fix: force the root <svg> element to always have
    width='256' height='256' viewBox='0 0 256 256'.

    The model often generates viewBox='0 0 24 24' (icon format) or
    viewBox='0.0 0.0 200.0 200.0'. The scorer renders to 256x256 pixels,
    so wrong viewBox scales the content to fill differently, killing SSIM.
    """
    # Replace viewBox with any value
    svg = re.sub(r'\bviewBox=["\'][^"\'>]*["\']', "viewBox='0 0 256 256'", svg, count=1)
    svg = re.sub(r'\bwidth=["\'][^"\'>]*["\']', "width='256'", svg, count=1)
    svg = re.sub(r'\bheight=["\'][^"\'>]*["\']', "height='256'", svg, count=1)
    # Add missing attributes
    if 'viewBox' not in svg[:300]:
        svg = svg.replace('<svg', "<svg viewBox='0 0 256 256'", 1)
    if 'width' not in svg[:300]:
        svg = svg.replace('<svg', "<svg width='256'", 1)
    if 'height' not in svg[:300]:
        svg = svg.replace('<svg', "<svg height='256'", 1)
    return svg


def strip_disallowed_tags(svg: str) -> str:
    """
    Instead of discarding the whole SVG when it contains one disallowed tag,
    try to strip just those tags and keep the rest.
    Handles the common case where the model adds a <script> or <image> tag.
    """
    # Remove self-closing disallowed tags: <script ... />
    svg = re.sub(r'<(?!/?(?:' + '|'.join(ALLOWED_TAGS) + r')\b)[^>]*/>', '', svg)
    # Remove paired disallowed tags and their content (script, image, etc.)
    for tag in ['script', 'image', 'foreignObject', 'iframe', 'video', 'audio']:
        svg = re.sub(rf'<{tag}[\s\S]*?</{tag}>', '', svg, flags=re.IGNORECASE)
        svg = re.sub(rf'<{tag}[^>]*/>', '', svg, flags=re.IGNORECASE)
    return svg


def is_valid_svg(svg: str) -> bool:
    if not svg or len(svg) > 16_000:
        return False
    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        return False
    path_count = 0
    for el in root.iter():
        tag = el.tag.split('}')[-1] if '}' in el.tag else el.tag
        if tag not in ALLOWED_TAGS:
            return False
        if tag == 'path':
            path_count += 1
    return path_count <= 256


def clean_svg(svg: str) -> str:
    svg = svg.strip()
    if not svg.startswith('<svg'):
        m = re.search(r'<svg', svg, re.IGNORECASE)
        if m:
            svg = svg[m.start():]
        else:
            return FALLBACK_SVG
    if 'xmlns' not in svg[:200]:
        svg = svg.replace('<svg', "<svg xmlns='http://www.w3.org/2000/svg'", 1)
    return svg


def truncate_svg(svg: str, max_len: int = 15_800) -> str:
    if len(svg) <= max_len:
        return svg
    cut = svg.rfind('<', 0, max_len)
    return svg[:cut] + '</svg>'


def extract_svg(text: str) -> str:
    m = SVG_REGEX.search(text)
    if m:
        return m.group(0).strip()
    # Partial SVG (model hit token limit mid-element)
    m2 = re.search(r'<svg[^>]*>[\s\S]*', text, re.IGNORECASE)
    if m2:
        partial = m2.group(0).rstrip()
        if not partial.lower().endswith('</svg>'):
            partial += '</svg>'
        return partial
    return FALLBACK_SVG


def repair_and_validate(svg: str) -> str:
    """
    Full repair pipeline. Returns a valid SVG or FALLBACK_SVG.
    Steps:
      1. Clean (find <svg start, add xmlns)
      2. Strip disallowed tags instead of discarding
      3. Normalize root attrs to 256x256
      4. Truncate to 15800 chars
      5. Validate; return fallback only if still broken
    """
    svg = clean_svg(svg)
    if svg == FALLBACK_SVG:
        return FALLBACK_SVG
    svg = strip_disallowed_tags(svg)
    svg = normalize_svg_root(svg)       # <-- fixes the 24x24 / 200x200 viewBox bug
    svg = truncate_svg(svg)
    if is_valid_svg(svg):
        return svg
    # Last attempt: try removing the last element in case truncation left broken XML
    last_open = svg.rfind('<', 0, len(svg) - 10)
    if last_open > 0:
        candidate = svg[:last_open] + '</svg>'
        if is_valid_svg(candidate):
            return candidate
    return FALLBACK_SVG


# Quick self-test
_t1 = repair_and_validate('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" width="24" height="24"><circle cx="12" cy="12" r="10" fill="blue"/></svg>')
assert 'viewBox=\'0 0 256 256\'' in _t1 or 'viewBox="0 0 256 256"' in _t1, f'viewBox fix failed: {_t1[:200]}'
print('SVG repair pipeline: OK')
print('Fixed viewBox sample:', _t1[:120])


SVG repair pipeline: OK
Fixed viewBox sample: <svg xmlns="http://www.w3.org/2000/svg" viewBox='0 0 256 256' width='256' height='256'><circle cx="12" cy="12" r="10" fi


In [5]:
test_df = pd.read_csv(Path(CONFIG['competition_data_dir']) / 'test.csv')
assert list(test_df.columns) == ['id', 'prompt']
print('test rows:', len(test_df))

# Show prompt token length distribution — critical to verify max_input_length
sample_lens = [len(p) for p in test_df['prompt'].tolist()]
print(f'Prompt char lengths — mean: {sum(sample_lens)/len(sample_lens):.0f}  '
      f'max: {max(sample_lens)}  min: {min(sample_lens)}')


test rows: 1000
Prompt char lengths — mean: 119  max: 488  min: 12


In [6]:
# ── System prompt — must exactly match v5 training ─────────────────────────
SYSTEM_PROMPT = (
    'You are an expert SVG artist and code generator. '
    'Given a natural-language description, output ONLY valid, visually accurate SVG code '
    'that faithfully represents the described scene or object. '
    'Requirements:\n'
    "- Root element: <svg xmlns='http://www.w3.org/2000/svg' width='256' height='256' viewBox='0 0 256 256'>\n"
    '- Use appropriate colors, shapes, and proportions to match the description visually\n'
    '- Fill the 256x256 canvas meaningfully — avoid large empty areas\n'
    '- Use multiple shapes (rect, circle, ellipse, path, polygon) layered for detail\n'
    '- Use fill and stroke colors that match the described appearance\n'
    '- Output SVG directly — no explanation, no markdown fences, no extra text'
)


def build_inference_prompt(prompt: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{prompt}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


# Verify token lengths so we know max_input_length=512 is sufficient
# (do this AFTER loading the tokenizer — re-run this cell if needed)
print('System prompt chars:', len(SYSTEM_PROMPT))
print('Full prompt for first test row:')
p0 = build_inference_prompt(test_df['prompt'].iloc[0])
print(f'  {len(p0)} chars')


System prompt chars: 676
Full prompt for first test row:
  808 chars


In [7]:
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_dir'], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_dir'],
    torch_dtype=torch.float16,
    device_map='balanced',
    trust_remote_code=True,
)
model.eval()
if hasattr(model, 'generation_config'):
    model.generation_config.max_length = None

first_device = next(model.parameters()).device
device_usage = {}
for _, param in model.named_parameters():
    d = str(param.device)
    device_usage[d] = device_usage.get(d, 0) + param.numel()
for dev, params in sorted(device_usage.items()):
    print(f'  {dev}: {params/1e9:.2f}B params')

# ── IMPORTANT: verify max_input_length is enough for the actual token lengths
sample_formatted = [build_inference_prompt(p) for p in test_df['prompt'].head(50).tolist()]
sample_enc = tokenizer(sample_formatted, padding=False, truncation=False)
tok_lens = [len(ids) for ids in sample_enc['input_ids']]

print(f'\nPrompt token lengths (first 50) — mean: {sum(tok_lens)/len(tok_lens):.0f}  '
      f'max: {max(tok_lens)}  min: {min(tok_lens)}')
if max(tok_lens) > CONFIG['max_input_length']:
    print(f'WARNING: some prompts exceed max_input_length={CONFIG["max_input_length"]}!')
    print(f'Consider increasing to {max(tok_lens) + 20}')
else:
    print(f'OK: all prompts fit within max_input_length={CONFIG["max_input_length"]}')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  cuda:0: 0.75B params
  cuda:1: 0.80B params

Prompt token lengths (first 50) — mean: 200  max: 251  min: 184
OK: all prompts fit within max_input_length=512


In [8]:
# ── Smoke test — checks both generation AND the repair pipeline ─────────────
@torch.no_grad()
def generate_batch(prompts: list) -> list:
    formatted = [build_inference_prompt(p) for p in prompts]
    enc = tokenizer(
        formatted,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=CONFIG['max_input_length'],
    )
    prompt_lens = enc['attention_mask'].sum(dim=1).tolist()
    enc = {k: v.to(first_device) for k, v in enc.items()}

    out_ids = model.generate(
        input_ids=enc['input_ids'],
        attention_mask=enc['attention_mask'],
        max_new_tokens=CONFIG['max_new_tokens'],
        do_sample=True,
        temperature=0.05,
        top_p=0.95,
        repetition_penalty=1.05,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    results = []
    for ids, plen in zip(out_ids, prompt_lens):
        decoded = tokenizer.decode(ids[int(plen):], skip_special_tokens=True)
        raw_svg = extract_svg(decoded)
        svg = repair_and_validate(raw_svg)
        results.append(svg)
    return results


print('Smoke test...')
smoke_prompts = [
    'a simple blue circle on a white background',
    'a red square with a yellow star inside',
]
smoke = generate_batch(smoke_prompts)
for p, s in zip(smoke_prompts, smoke):
    print(f'Prompt: {p}')
    print(f'  Length: {len(s)} chars | Valid: {is_valid_svg(s)}')
    print(f'  Start : {s[:150]}')
    print()


Smoke test...
Prompt: a simple blue circle on a white background
  Length: 248 chars | Valid: True
  Start : <svg height='256' viewBox='0 0 256 256' width='256' xmlns="http://www.w3.org/2000/svg"><path d="m12 2c-5.52 0-10 4.48-10 10s4.48 10 10 10 10-4.48 10-1

Prompt: a red square with a yellow star inside
  Length: 491 chars | Valid: True
  Start : <svg height='256' viewBox='0 0 256 256' width='256' xmlns="http://www.w3.org/2000/svg"><path d="m27 2h-22c-2.21 0-4 1.79-4 4v22c0 2.21 1.79 4 4 4h22c2



In [9]:
from tqdm.auto import tqdm

prompts_list = test_df['prompt'].tolist()
ids_list     = test_df['id'].tolist()
bs           = CONFIG['inference_batch_size']

all_svgs = []
fallback_count = 0
wrong_viewbox_fixed = 0

for start in tqdm(range(0, len(prompts_list), bs), desc='Generating SVGs'):
    batch_prompts = prompts_list[start : start + bs]

    formatted = [build_inference_prompt(p) for p in batch_prompts]
    enc = tokenizer(
        formatted,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=CONFIG['max_input_length'],
    )
    prompt_lens = enc['attention_mask'].sum(dim=1).tolist()
    enc = {k: v.to(first_device) for k, v in enc.items()}

    with torch.no_grad():
        out_ids = model.generate(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            max_new_tokens=CONFIG['max_new_tokens'],
            do_sample=True,
            temperature=0.05,
            top_p=0.95,
            repetition_penalty=1.05,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    for ids, plen in zip(out_ids, prompt_lens):
        decoded = tokenizer.decode(ids[int(plen):], skip_special_tokens=True)
        raw_svg = extract_svg(decoded)

        # Track whether the viewBox needed fixing
        had_wrong_vb = raw_svg != FALLBACK_SVG and (
            'viewBox' not in raw_svg[:300]
            or "256 256" not in raw_svg[:300]
        )

        svg = repair_and_validate(raw_svg)

        if svg == FALLBACK_SVG:
            fallback_count += 1
        elif had_wrong_vb:
            wrong_viewbox_fixed += 1

        all_svgs.append(svg)

    if (start // bs) % 10 == 0 and start > 0:
        gc.collect()
        torch.cuda.empty_cache()

print(f'Generated          : {len(all_svgs)}')
print(f'Fallbacks          : {fallback_count} ({100*fallback_count/len(all_svgs):.1f}%)')
print(f'viewBox fixed      : {wrong_viewbox_fixed}')

svg_lens = [len(s) for s in all_svgs]
print(f'SVG chars — mean: {sum(svg_lens)/len(svg_lens):.0f}  '
      f'median: {sorted(svg_lens)[len(svg_lens)//2]:.0f}  '
      f'min: {min(svg_lens)}  max: {max(svg_lens)}')

# ── Diagnose remaining fallbacks ──────────────────────────────────────────
print('\n--- Fallback diagnosis (first 3 failures) ---')
fail_count = 0
for i, (svg, prompt) in enumerate(zip(all_svgs, prompts_list)):
    if svg == FALLBACK_SVG and fail_count < 3:
        print(f'Row {i}: prompt[:60]={prompt[:60]!r}')
        fail_count += 1


Generating SVGs:   0%|          | 0/167 [00:00<?, ?it/s]

Generated          : 1000
Fallbacks          : 11 (1.1%)
viewBox fixed      : 989
SVG chars — mean: 1045  median: 1049  min: 93  max: 3159

--- Fallback diagnosis (first 3 failures) ---
Row 120: prompt[:60]='A line drawing of a closed cardboard box.'
Row 132: prompt[:60]='A black and white line drawing of a bunch of grapes.'
Row 151: prompt[:60]='A line drawing of a notebook with a silhouette of a person i'


In [10]:
submission = pd.DataFrame({'id': ids_list, 'svg': all_svgs})
submission.to_csv(CONFIG['submission_path'], index=False)
print('Saved:', CONFIG['submission_path'])
print(f'Rows: {len(submission)}')

# Verify first few SVGs look correct
for i in range(3):
    s = all_svgs[i]
    print(f'Row {i}: {len(s)} chars | valid={is_valid_svg(s)} | start={s[:100]}')


Saved: /kaggle/working/submission.csv
Rows: 1000
Row 0: 2163 chars | valid=True | start=<svg height='256' viewBox='0 0 256 256' width='256' xmlns="http://www.w3.org/2000/svg"><path d="m114
Row 1: 1432 chars | valid=True | start=<svg xmlns="http://www.w3.org/2000/svg" viewBox='0 0 256 256' height='256' width='256'><path fill="#
Row 2: 1419 chars | valid=True | start=<svg xmlns="http://www.w3.org/2000/svg" viewBox='0 0 256 256' height='256' width='256'><path fill="#
